# Urban Mobility and Economic Productivity in Latin American Cities

Is congestion associated with GDP per capita across Latin American cities? This notebook combines the TomTom Traffic Index with OECD city indicators for a single year, builds a merged dataset and explores the relationship between mobility and economic output.

## Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

TRAFFIC_PATH = Path("data/tomtom_traffic.csv")
ECONOMY_PATH = Path("data/oecd_city_economy.csv")
OUTPUT_DIR = Path("outputs")
YEAR = 2024

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Load data

In [ ]:
traffic = pd.read_csv(TRAFFIC_PATH)
economy = pd.read_csv(ECONOMY_PATH)

print(f"Traffic: {traffic.shape[0]:,} rows | Economy: {economy.shape[0]:,} rows")

In [ ]:
traffic.head()

In [ ]:
economy.head()

In [ ]:
traffic.info()

In [ ]:
economy.info()

## Data preparation

### Traffic

Column names are standardized to `snake_case`, timestamps are parsed and the year is extracted for filtering.

In [ ]:
TRAFFIC_COLUMNS = {
    "Country": "country",
    "City": "city",
    "UpdateTimeUTC": "update_time_utc",
    "JamsDelay": "jams_delay",
    "TrafficIndexLive": "traffic_index_live",
    "JamsLengthInKms": "jams_length_in_kms",
    "JamsCount": "jams_count",
    "TrafficIndexWeekAgo": "traffic_index_week_ago",
    "UpdateTimeUTCWeekAgo": "update_time_utc_week_ago",
    "TravelTimeLivePer10KmsMins": "travel_time_live_per_10kms_mins",
    "TravelTimeHistoricPer10KmsMins": "travel_time_historic_per_10kms_mins",
    "MinsDelay": "mins_delay",
}

traffic = traffic.rename(columns=TRAFFIC_COLUMNS)
for col in ("update_time_utc", "update_time_utc_week_ago"):
    traffic[col] = pd.to_datetime(traffic[col], errors="coerce")
traffic["year"] = traffic["update_time_utc"].dt.year

traffic.info()

### Economy

Indicators arrive as text with decimal commas, thousands separators and `%` symbols, so they are converted to floats.

In [ ]:
ECONOMY_COLUMNS = {
    "Year": "year",
    "City": "city",
    "Country": "country",
    "City GDP/capita": "city_gdp_capita",
    "Unemployment %": "unemployment_pct",
    "PM2.5 (μg/m³)": "pm25",
    "Population (M)": "population_m",
}


def to_float(series, thousands=None, strip=""):
    """Parse text numbers that use a decimal comma and an optional thousands separator."""
    cleaned = series.astype(str)
    if strip:
        cleaned = cleaned.str.replace(strip, "", regex=False)
    if thousands:
        cleaned = cleaned.str.replace(thousands, "", regex=False)
    return cleaned.str.replace(",", ".", regex=False).astype(float)


economy = economy.rename(columns=ECONOMY_COLUMNS)
economy["city_gdp_capita"] = to_float(economy["city_gdp_capita"], thousands=".")
economy["unemployment_pct"] = to_float(economy["unemployment_pct"], strip="%")
economy["pm25"] = to_float(economy["pm25"])
economy["population_m"] = to_float(economy["population_m"])
economy["population"] = economy["population_m"] * 1_000_000

economy.info()

## Yearly traffic averages by city

In [ ]:
traffic_year = (
    traffic[traffic["year"] == YEAR]
    .groupby(["city", "country", "year"], as_index=False)
    .agg(
        jams_delay=("jams_delay", "mean"),
        traffic_index_live=("traffic_index_live", "mean"),
        jams_length_kms=("jams_length_in_kms", "mean"),
        jams_count=("jams_count", "mean"),
        mins_delay=("mins_delay", "mean"),
        travel_time_live_per_10kms_mins=("travel_time_live_per_10kms_mins", "mean"),
        travel_time_hist_per_10kms_mins=("travel_time_historic_per_10kms_mins", "mean"),
    )
)

traffic_year.sort_values("jams_delay", ascending=False).head(10)

## Merge mobility and economy

An inner join keeps only the cities present in both sources.

In [ ]:
economy_cols = ["city", "year", "city_gdp_capita", "unemployment_pct", "pm25", "population"]
economy_year = economy.loc[economy["year"] == YEAR, economy_cols]

merged = traffic_year.merge(economy_year, on=["city", "year"], how="inner")

print(f"{len(merged)} cities in both datasets")
merged

## Exploratory analysis

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(x=merged["jams_delay"], showmeans=True, ax=ax)
ax.set_xlabel("Total congestion delay (minutes)")
ax.set_title(f"Jams delay distribution ({YEAR})\nMean: {merged['jams_delay'].mean():.2f}")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "jams_delay_boxplot.png", dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(merged["city_gdp_capita"], bins=8, edgecolor="black")
ax.set_title(f"City GDP per capita distribution ({YEAR})")
ax.set_xlabel("GDP per capita (USD)")
ax.set_ylabel("Number of cities")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "gdp_per_capita_hist.png", dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
merged.plot(kind="bar", x="city", y=["jams_delay", "city_gdp_capita"], ax=ax)
ax.set_title(f"Jams delay and GDP per capita by city ({YEAR})")
ax.set_xlabel("City")
ax.set_ylabel("Value")
ax.tick_params(axis="x", rotation=90)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "jams_delay_vs_gdp.png", dpi=150)
plt.show()

### Correlations

Pearson correlations of the congestion metrics with GDP per capita and population. With a sample this small, they should be read as descriptive and not as evidence of a relationship.

In [ ]:
targets = ["jams_delay", "traffic_index_live"]
drivers = ["city_gdp_capita", "population"]

merged[targets + drivers].corr().loc[targets, drivers].round(2)

## Export

In [ ]:
output_path = OUTPUT_DIR / f"ladb_mobility_economy_{YEAR}_clean.csv"
merged.to_csv(output_path, index=False)
print(f"Saved {output_path}")

## Limitations

- Only a small number of cities remain after the join, and most are Brazilian, so correlations are unstable.
- Outliers in the economic indicators (for example, an implausible GDP per capita for a single city) can drive the results and should be verified against the source.
- Association between congestion and GDP does not support conclusions about the return on transport investment.